<font color=#009afa size=50>DAT5001 - Visualisation of Data</font>

<font size=50>1 - Cleaning Satellite Dataset for Presentation</font>

|**Assessment Code:** |PRES1|
|---|---|
|**Author:**|Tom Rowan, ST20285213|
|**Course:**| DAT5001_S2_25|
|**Course Tutor:**|Dr. Imtiaz Khan|


## Initialise Notebook

In [51]:
# Install Requirements Using Pip
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


### Import Python Libraries

In [52]:
# Standard Libraries
from datetime import datetime
import sys
import time

# Pandas and numpy
import pandas as pd
import numpy as np

# Tabulate
from tabulate import tabulate

# Colorama
from colorama import Fore, Back, Style

# Requests
import requests

# Pycountry 
import pycountry

# GeoPy
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

# Do not print warnings!
import warnings
warnings.filterwarnings('ignore')

### Check for Internet Access

In [53]:
# Function to print coloured output
def print_output (str1, str2, delim=':', colour='yellow'):
    if colour.upper() in dir(Fore):
        colour = colour.upper()
    else:
        colour = 'YELLOW'
    print (f'{getattr(Fore, colour)}{str(str1)}{delim}{Style.RESET_ALL} {str(str2)}')

In [54]:
# Coding Syntax Note: _variable_name --> temporary / locally used variable, discarded after use.

# Check for Internet Connectivity
def check_internet (url_list):
    online = False
    for _url in url_list:
        try:
            response = requests.get(_url, timeout=3)
            if response.status_code == 200:
                online = True
                break
        except requests.RequestException:
            output_string = f"Check Internet failed for {_url}"
            print_output ('[FAIL]',output_string, colour='red', delim='')
    return online


_urls_to_check = ['http://connectivitycheck.gstatic.com/generate_204',
                  'https://www.planet4589.org',
                  'https://datascience.daemonchild.com']

if check_internet (_urls_to_check ):
    print_output ('[INTERNET]','Internet Connectivity Detected. Using Internet resources.',colour='green', delim='')
    online = True

else:
    print ()
    print_output ('[INTERNET]','No Internet Connectivity.', colour='red', delim='')
    sys.exit('Stopping notebook running here.')


[INTERNET] Internet Connectivity Detected. Using Internet resources.


## Load the Datasets from the Internet

Reference Website:
https://www.planet4589.org/space/gcat/index.html

Tab Separated Value Files: 

- SatCat: https://www.planet4589.org/space/gcat/tsv/cat/satcat.tsv
- Events: https://planet4589.org/space/gcat/tsv/cat/ecat.tsv
- Payload: https://www.planet4589.org/space/gcat/tsv/cat/psatcat.tsv   
- Org Data: https://www.planet4589.org/space/gcat/tsv/tables/orgs.tsv
- Launch sites: https://www.planet4589.org/space/gcat/tsv/tables/sites.tsv

Documentation: https://www.planet4589.org/space/gcat/pdf/doc.pdf

Reference: "McDowell, J, 2020: General Catalog of Artificial Space Objects, https://planet4589.org/space/gcat", First accessed 24/10/2025

> VIII.2.1.1 The Standard (S) Satellite Catalog
>
> Since the late 1950s the US government has maintained a catalog of satellites (often referred to as the SATCAT). Each
satellite is given a sequential number (1 was the rocket stage from the first Sputnik). These numbers are sometimes
referred to as SSN numbers.

In [55]:
# This will get the very latest dataset from John McDowells's page. This dataset is updated daily!
satcat_df = pd.read_csv('https://www.planet4589.org/space/gcat/tsv/cat/satcat.tsv', sep='\t')
events_df = pd.read_csv('https://planet4589.org/space/gcat/tsv/cat/ecat.tsv', sep='\t')
pay_df = pd.read_csv('https://www.planet4589.org/space/gcat/tsv/cat/psatcat.tsv', sep='\t')
orgs_df = pd.read_csv('https://www.planet4589.org/space/gcat/tsv/tables/orgs.tsv', sep='\t')
lsite_df = pd.read_csv('https://www.planet4589.org/space/gcat/tsv/tables/sites.tsv', sep='\t')

datasets = {'satcat_df': satcat_df, 
            'events_df': events_df, 
            'pay_df': pay_df, 
            'orgs_df': orgs_df,
            'lsite_df': lsite_df
            }

Save a local copy

In [56]:
# Generate Timestamp
today = pd.Timestamp(datetime.today())
timestamp_formatted = today.strftime("%Y%m%d")

# Create filenames, timestamped
filepath = '../../datasets/'

for k,v in datasets.items():
    filename = f'{filepath}{k}_{timestamp_formatted}.csv'
    v.to_csv(filename, index=False)


### Explore these datasets

In [57]:
for k,v in datasets.items():
    print ("-----")
    print ("Dataset: " + k)
    print ("")
    print (v.columns)
    print ("")
    print (v.shape)

-----
Dataset: satcat_df

Index(['#JCAT', 'Satcat', 'Launch_Tag', 'Piece', 'Type', 'Name', 'PLName',
       'LDate', 'Parent', 'SDate', 'Primary', 'DDate', 'Status', 'Dest',
       'Owner', 'State', 'Manufacturer', 'Bus', 'Motor', 'Mass', 'MassFlag',
       'DryMass', 'DryFlag', 'TotMass', 'TotFlag', 'Length', 'LFlag',
       'Diameter', 'DFlag', 'Span', 'SpanFlag', 'Shape', 'ODate', 'Perigee',
       'PF', 'Apogee', 'AF', 'Inc', 'IF', 'OpOrbit', 'OQUAL', 'AltNames'],
      dtype='object')

(66781, 42)
-----
Dataset: events_df

Index(['#JCAT', 'Satcat', 'Launch_Tag', 'Piece', 'Type', 'Name', 'PLName',
       'LDate', 'Parent', 'SDate', 'Primary', 'DDate', 'Status', 'Dest',
       'Owner', 'State', 'Manufacturer', 'Bus', 'Motor', 'Mass', 'MassFlag',
       'DryMass', 'DryFlag', 'TotMass', 'TotFlag', 'Length', 'LFlag',
       'Diameter', 'DFlag', 'Span', 'SpanFlag', 'Shape', 'ODate', 'Perigee',
       'PF', 'Apogee', 'AF', 'Inc', 'IF', 'OpOrbit', 'OQUAL', 'AltNames'],
      dtype='object

## Data Cleaning
Remove columns that are not useful for the report.

**Satcat Database**

In [58]:
# Remove columns we can't use
columns_to_keep = ['Piece', 'Type', 'Name', 'LDate', 'DDate', 'Status', 'Owner', 'State', 'Manufacturer', 'Mass', 'Length', 'Diameter', 'Span', 'Shape', 'Perigee', 'Apogee']
sdf = satcat_df[columns_to_keep]

# Rename Columns for easier reading
sdf.rename(columns={'Piece':'CosparID', 'LDate':'Launch Date', 'DDate':'Decay Date', 'Owner':'Operator Code','Manufacturer':'Manufacturer Code'}, inplace=True)

The date fields are a little challenging. Some contain just dates (US format), some contain date and times, and some contain "-" or " -". These are fixed now because other calculations will rely on solid dates.

In [59]:
sdf['Launch Date'] = sdf['Launch Date'].replace("-", np.nan)
sdf['Decay Date'] = sdf['Decay Date'].replace("-", np.nan)
sdf['Decay Date'] = sdf['Decay Date'].replace(" -", np.nan)

# Strip trailing time values: keep only first 3 tokens (year, month, day)
sdf['Launch Date'] = (
    sdf['Launch Date']
    .str.split()        # split into parts
    .str[:3]            # keep first 3
    .str.join(" ")      # join back into a string
)

sdf['Decay Date'] = (
    sdf['Decay Date']
    .str.split()        # split into parts
    .str[:3]            # keep first 3# Replace "-" with NaN

    .str.join(" ")      # join back into a string
)

# There are some empty rows - such as where JCAT = S49051
sdf = sdf.dropna(subset=['Launch Date'])


# There are some dates that are questionable - remove the question marks
sdf['Decay Date'] = sdf['Decay Date'].str.replace('?', '', regex=False)

In [60]:
sdf.shape

(66719, 16)

**Events Database**

In [61]:
edf_columns_to_keep = ['Piece', 'LDate', 'DDate', 'Status', 'Perigee', 'Apogee']
edf = events_df[edf_columns_to_keep]

edf.rename(columns={'Piece':'CosparID', 'LDate':'Launch Date', 'DDate':'Decay Date'}, inplace=True)

In [62]:
edf['Decay Date'] = edf['Decay Date'].replace("-", np.nan)
edf['Decay Date'] = edf['Decay Date'].replace(" -", np.nan)

# Strip trailing time values: keep only first 3 tokens (year, month, day)
edf['Launch Date'] = (
    edf['Launch Date']
    .str.split()        # split into parts
    .str[:3]            # keep first 3
    .str.join(" ")      # join back into a string
)

edf['Decay Date'] = (
    edf['Decay Date']
    .str.split()        # split into parts
    .str[:3]            # keep first 3# Replace "-" with NaN

    .str.join(" ")      # join back into a string
)

# There are some dates that are questionable - remove the question marks
edf['Decay Date'] = edf['Decay Date'].str.replace('?', '', regex=False)

**Organisations Database**

In [63]:
# Create the CombinedName column using owner_EName if available, otherwise owner_Name
orgs_df['CombinedName'] = orgs_df['EName'].fillna(orgs_df['Name'])

# Remove columns we can't use
orgs_columns_to_keep = ['#Code', 'ShortName', 'CombinedName', 'Longitude', 'Latitude', 'Parent']
odf = orgs_df[orgs_columns_to_keep]

# Rename Columns for easier reading
odf.rename(columns={'#Code':'Org Code', 'ShortName':'Short Name', 'CombinedName':'Long Name'}, inplace=True)

Convert Latitude and Longitude to City and State values

In [64]:
# Do this very rarely! It takes three hours due to 2 second API rate limits.
do_lat_lon_lookup = False

In [65]:
# Extract data from Nominatim database
# Ratelimited to 1 second, using 1.5 for safety

rate_limit_time = 1.5

def get_location_info(lat, lon):
    time.sleep(rate_limit_time)
    lat_lon = f'{lat},{lon}'
    
    try:
        location = geolocator.reverse(lat_lon, language="english")
        if location and location.raw.get('address'):
            address = location.raw['address']
            city = address.get('city', address.get('town', address.get('village', '')))
            state = address.get('state', '')
            country = address.get('country', '')
            countrycode = address.get('country_code', '')

            return f"{city}|{state}|{country}|{countrycode}"

    except GeocoderTimedOut:
        print("Error: Geocoding timed out. Please try again later.")
    except GeocoderServiceError as e:
        print(f"Service error occurred: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None
    
if do_lat_lon_lookup and online:
    print_output ('[CITY/STATE/COUNTRY DATA]','This is going to take three hours.',colour='green', delim='')
    
    # Set up the geocoder
    geolocator = Nominatim(user_agent="Cardiff_Met_Project_Tom_Rowan")

    # Apply the function using lambda
    odf['GeoInfo'] = odf.apply(lambda row: get_location_info(row['Latitude'], row['Longitude']), axis=1)

    odf[['City', 'State', 'Country', 'Country Code']] = odf['GeoInfo'].str.split('|', expand=True)
    odf.drop(columns=['GeoInfo'], axis=1, inplace=True)
    odf.to_csv('../../datasets/odf-complete-version.csv', index=False)
else:
    print_output ('[CITY/STATE/COUNTRY DATA]','Loading from saved version',colour='green', delim='')
    odf = pd.read_csv('../../datasets/odf-complete-version.csv')
    #odf.drop(columns=['GeoInfo'], axis=1, inplace=True)
    #odf.to_csv('../../datasets/odf-complete-version.csv', index=False)

[CITY/STATE/COUNTRY DATA] Loading from saved version


In [66]:
odf.shape

(3968, 10)

In [67]:
odf.sample(20)

,Org Code,Short Name,Long Name,Longitude,Latitude,Parent,City,State,Country,Country Code
3097,SHINSHU,Shinshu U,Shinshu University,138.1800,36.6300,-,長野市,NaN,日本,jp
2826,QTT,QuanTuTong,Full Map Network Co.,116.4900,39.8000,-,大兴区,北京市,中国,cn
432,AFRLH,AFRL-VS/Hanscom,-,-71.2800,42.4900,AFRL,Bedford,Massachusetts,United States of America,us
3706,USSPC2,USSPACECOM,-,-104.7000,38.8200,DOD,Colorado Springs,Colorado,United States of America,us
3930,ZDNC,Zhongdian Nong.,China Power Investment New Agricultural Innov....,116.3000,39.9600,-,海淀区,北京市,中国,cn
3718,UTCWL,UTC Aerospace,-,-72.6500,41.9300,UTC2,Windsor Locks,Connecticut,United States of America,us
3651,UPM,IDR/UPM,Polytechnic U. of Madrid,-3.7300,40.4500,-,Madrid,Comunidad de Madrid,España,es
509,AMDL,AMSAT-D,AMSAT-Germany,8.7700,50.8100,AMSAT,Marburg,Hessen,Deutschland,de
1403,GATOM,General Atomics,-,-117.1600,32.7200,-,San Diego,California,United States of America,us
867,CASC1,CASC,China Aerospace Sci/Tech Corp,116.3800,39.9200,-,西城区,北京市,中国,cn


**Payload Database**

In [68]:
payload_columns_to_keep = ['Piece', 'Program', 'Class', 'Category']
pdf = pay_df[payload_columns_to_keep]

# Rename 'Piece' to 'CosparID'
pdf.rename(columns={'Piece':'CosparID'}, inplace=True)

## Combine the Datasets

In [69]:
# Rename odf columns for Operator merge
owner_odf = odf.rename(columns=lambda x: f'Operator {x}' if x != 'Org Code' else 'Operator Code')
sdf = sdf.merge(owner_odf, on='Operator Code', how='left')

# Repeat for Manufacturer merge
manufacturer_odf = odf.rename(columns=lambda x: f'Manufacturer {x}' if x != 'Org Code' else 'Manufacturer Code')
sdf = sdf.merge(manufacturer_odf, on='Manufacturer Code', how='left')

# Merge payload data
sdf = sdf.merge(pdf, on='CosparID', how='left')

In [70]:
sdf.columns

Index(['CosparID', 'Type', 'Name', 'Launch Date', 'Decay Date', 'Status',
       'Operator Code', 'State', 'Manufacturer Code', 'Mass', 'Length',
       'Diameter', 'Span', 'Shape', 'Perigee', 'Apogee', 'Operator Short Name',
       'Operator Long Name', 'Operator Longitude', 'Operator Latitude',
       'Operator Parent', 'Operator City', 'Operator State',
       'Operator Country', 'Operator Country Code', 'Manufacturer Short Name',
       'Manufacturer Long Name', 'Manufacturer Longitude',
       'Manufacturer Latitude', 'Manufacturer Parent', 'Manufacturer City',
       'Manufacturer State', 'Manufacturer Country',
       'Manufacturer Country Code', 'Program', 'Class', 'Category'],
      dtype='object')

### Using Events to Update the Dataset

In [71]:
sdf['Decay Date'].isna().value_counts()

Decay Date
False    35277
True     31442
Name: count, dtype: int64

The events database contains updates to the Satcat data, so these need to be applied like a database log to get the updated values.

For example, this ID shows as being docked.

In [72]:
sdf[sdf['CosparID'] == '1988-038A']['Status']

19116    DK
Name: Status, dtype: object

However, later on in the events database, it undocked (UDK) and then finally deorbited (D):

In [73]:
edf[edf['CosparID'] == '1988-038A']['Status']

767    UDK
770      D
Name: Status, dtype: object

In [74]:
# Ensure types are comparible
sdf['Launch Date'] = pd.to_datetime(sdf['Launch Date'], errors='coerce')
sdf['Decay Date'] = pd.to_datetime(sdf['Decay Date'], errors='coerce')

edf['Launch Date'] = pd.to_datetime(edf['Launch Date'], errors='coerce')
edf['Decay Date'] = pd.to_datetime(edf['Decay Date'], errors='coerce')

# Dedupe edf to the latest update per CosparID by index, therefore keeping the last update.
edf_latest = (
    edf.sort_index()                # ensure index order is respected
       .drop_duplicates('CosparID', keep='last')
)

# Merge and update only specified columns
merged = sdf.merge(edf_latest, on='CosparID', how='left', suffixes=('', '_update'))

# Columns to update from edf_latest
cols_to_update = ['Launch Date', 'Decay Date', 'Status', 'Perigee', 'Apogee']

for col in cols_to_update:
    update_col = f"{col}_update"
    if update_col in merged.columns:
        merged[col] = merged[update_col].combine_first(merged[col])

# Clean up _update columns
sdf = merged.drop(columns=[c for c in merged.columns if c.endswith('_update')])



Check with the example from above:

In [75]:
sdf[sdf['CosparID'] == '1988-038A']['Status']

19116    D
Name: Status, dtype: object

# Feature Engineering

### Category, Class and Type Fields

In [76]:
# These mappings were created from the documentation:

class_json = {
  "A": "Non-commercial",        # Academic / Amateur / Non-Profit
  "B": "Commercial",
  "C": "Government (Civil)",
  "D": "Government (Defence)",  # and Military and Intelligence
  "U": "Unknown"
}

type_json = {
  "P": "Payload",
  "C": "Component",
  "R": "Rocket Body",
  "D": "Debris",
  "S": "Suborbital payload",
  "X": "Deleted Object",
  "Z": "Spurious Entry"
}

category_json = {
  "AST": "Astronomy",
  "BIO": "Biology",
  "CAL": "Calibration",
  "COM": "Communications",
  "EDU": "Education",
  "EOSCI": "Earth observing science",
  "EW": "Missile early warning",
  "GEOD": "Geodesy",
  "IMG": "Imaging (optical)",
  "IMG-R": "Imaging (radar)",
  "INF": "Infrastructure",
  "MET": "Meteorology (imaging)",
  "MET-RO": "Meteorology (radio occultation)",
  "MGRAV": "Microgravity experiments",
  "MISC": "Miscellaneous",
  "NAV": "Navigation",
  "PLAN": "Deep space related mission",
  "RB": "Rocket body",
  "RV": "Reentry vehicle",
  "SCI": "Scientific studies",
  "SIG": "Signals intelligence",
  "SS": "Spaceship",
  "TARG": "Target",
  "TECH": "Technology",
  "WEAPON": "Weapon",
  "UNK": "Unknown"
}

status_json = {
    "O": "In orbit",
    "AO": "Attached in orbit",
    "AO IN": "Attached inside",
    "UDK": "Undocked",
    "REL": "Released",
    "DEP": "Deployed",
    "TO": "Takeoff Launch",
    "TOA": "Attached at takeoff",
    "OX": "In orbit",
    "E": "Exploded",
    "N": "Renamed",
    "NA": "Renamed attached",
    "LEASE": "Lease",
    "S": "Targeted suborbital entry/impact",
    "R": "Reentered",
    "D": "Deorbited",
    "L": "Landed",
    "LF": "Failed landing",
    "AS": "Attached Suborbital reentered",
    "AR": "Reentered attached",
    "AR IN": "Reentered inside",
    "AL": "Landed attached",
    "AL IN": "Landed inside",
    "TX": "Transmission ended",
    "F": "Failed",
    "AF": "Failed attached",
    "DK": "Docked",
    "GRP": "Grappled",
    "ATT": "Attached",
    "TFR": "Transfer Inside",
    "TFR E": "Transfer external Inside",
    "TFR IN": "Transfer in"
}

In [77]:
# Split the Category column at the first '/'
split_cols = sdf['Category'].str.split('/', n=1, expand=True)

# Remove trailing spurious characters from each part using a regex
sdf['Category'] = sdf['Category'].fillna('UNK')
sdf['Primary Category'] = split_cols[0].str.strip().str.replace(r'[*?]+$', '', regex=True) if split_cols.shape[0] > 1 else 'UNK'
sdf['Secondary Category'] = split_cols[1].str.strip().str.replace(r'[*?]+$', '', regex=True) if split_cols.shape[1] > 1 else 'UNK'

# Type field - Remove Sub-Types
sdf['Type'] = sdf['Type'].str[0]

# Remove X and Z objects ("Spurious and unknown")
sdf = sdf[~sdf['Type'].isin(['X', 'Z'])]

# Set Class to Unknown if not labelled
sdf['Class'] = sdf['Class'].fillna('U')

# Map the descriptions for better data visualisation labelling
sdf['Class Desc'] = sdf['Class'].map(class_json)
sdf['Type Desc'] = sdf['Type'].map(type_json)
sdf['Primary Category Desc'] = sdf['Primary Category'].map(category_json)
sdf['Secondary Category Desc'] = sdf['Secondary Category'].map(category_json)
sdf['Status Desc'] = sdf['Status'].map(status_json)

Delete coded columns and then swap round so that "Type Desc" becomes just "Type"

In [78]:
sdf.drop(columns=['Type', 'Class', 'Category', 'Primary Category', 'Secondary Category', 'Status'], inplace=True, axis=1)
sdf.rename(columns={'Type Desc':'Type'}, inplace=True)

# Rename columns ending with " Desc"
sdf = sdf.rename(
    columns=lambda c: c[:-5] if c.endswith(" Desc") else c
)

In [79]:
sdf['Type'].value_counts()

Type
Debris         27825
Payload        24765
Component       7905
Rocket Body     6198
Name: count, dtype: int64

In [80]:
sdf['Class'].value_counts()

Class
Unknown                 41840
Commercial              15347
Government (Defence)     5420
Government (Civil)       3168
Non-commercial            897
Name: count, dtype: int64

In [81]:
sdf['Primary Category'].value_counts()

Primary Category
Communications                     15111
Imaging (optical)                   3052
Technology                          1607
Calibration                          811
Spaceship                            766
Signals intelligence                 708
Navigation                           624
Scientific studies                   524
Meteorology (imaging)                304
Imaging (radar)                      275
Deep space related mission           257
Astronomy                            219
Missile early warning                213
Earth observing science              120
Geodesy                               65
Meteorology (radio occultation)       63
Weapon                                37
Microgravity experiments              32
Biology                               30
Target                                21
Reentry vehicle                       10
Education                              4
Name: count, dtype: int64

In [82]:
sdf['Status'].value_counts()

Status
Reentered             32228
In orbit              31756
Landed                 1326
Deorbited               401
Reentered attached      300
Attached in orbit        87
Landed attached          24
Lease                    14
Exploded                 12
Landed inside             3
Transfer Inside           3
Released                  2
Reentered inside          1
Renamed                   1
Name: count, dtype: int64

### Is the object still in orbit?

Work out whether the object is still in orbit by combining specific status wording and whether there is a Decayed Date recorded or not.

In [83]:
# Define category lists
not_in_orbit_categories = ['Reentered', 'Landed', 'Deorbited', 'Reentered attached', 'Reentered inside', 'Landed attached', 'Landed inside']
in_orbit_categories = ['In Orbit', 'Attached in orbit', 'Docked']

# If not in any of these categories, but has a decay date
sdf['In Orbit'] = (
    ( ~sdf['Status'].isin(not_in_orbit_categories) & ~sdf['Status'].isin(in_orbit_categories) & sdf['Decay Date'].notna() )
)

# Create the new column 'In Orbit'
sdf['In Orbit'] = (
    sdf['Status'].isin(in_orbit_categories) |       # Found in "In Orbit category"
    (~sdf['Status'].isin(not_in_orbit_categories))  # Not found in "No In Orbit category"
)



Returning to the example from before:

In [84]:
sdf[sdf['CosparID'] == '1988-038A']['In Orbit']

19116    False
Name: In Orbit, dtype: bool

In [85]:
sdf.sample(20)[['CosparID','Type', 'Status', 'Launch Date', 'Decay Date','In Orbit']]

,CosparID,Type,Status,Launch Date,Decay Date,In Orbit
7184,1973-086EW,Debris,In orbit,1973-11-06,NaT,True
16591,1986-014B,Component,In orbit,1986-02-09,NaT,True
50595,1982-092AKU,Debris,Reentered,1982-09-16,2022-09-29,False
12619,1981-069C,Component,Reentered,1981-07-30,1981-07-31,False
33960,1997-051BW,Debris,Reentered,1997-09-14,2023-05-23,False
22091,1992-055C,Component,Reentered,1992-08-15,1992-08-17,False
34125,1993-036KX,Debris,Reentered,1993-06-16,2025-04-08,False
55949,2023-042M,Payload,Reentered,2023-03-24,2024-06-23,False
12551,1981-021D,Component,Reentered,1981-03-05,1981-07-19,False
47914,2021-021BH,Payload,Reentered,2021-03-14,2024-05-15,False


In [86]:
sdf['In Orbit'].value_counts().reset_index()

,In Orbit,count
0,False,34283
1,True,32410


In [87]:
# Check the results looks sane
filtered_df = sdf[(sdf['Type'] == 'Payload')]
                  
pivot_table = filtered_df.pivot_table(
    index='Status',
    columns='In Orbit',
    aggfunc='size',
    fill_value=0
)

# Display the result
print(pivot_table)

In Orbit            False  True 
Status                          
Attached in orbit       0     86
Deorbited             362      0
Exploded                0      8
In orbit                0  16845
Landed               1325      0
Landed attached        24      0
Landed inside           3      0
Lease                   0     14
Reentered            5455      0
Reentered attached    298      0
Released                0      1
Renamed                 0      1
Transfer Inside         0      1


### Generate Additional Date Values

The dates that are available are:

- Launch Date (always available)
- Decay Date (may be NULL or '?')

These values are converted from strings in the base dataset into python datetime objects.
This allows for comparisons and for additional values ot be inferred from them.

The following date columns were created:

- Launch Year
- Launch Month (Numerical)
- Launch Month (Text)
- Launch Decade (1950's: is 1950-1959.)
- Decay Year
- Decay Month (Numerical)
- Decay Month (Text)
- Decay Decade 

A temporary value for 'Effective Decay Date' is either copied from 'Decay Date' or set to 'Today' if the object is "In Orbit".
This allows for lifetime values in days, months and years to be calculated.

Finally, the datetime objects are reverted to strings (in UK format).

In [88]:
# Modify date columns
# Convert to datetime format
sdf['Launch Date'] = pd.to_datetime(sdf['Launch Date'], format='%Y %b %d', errors='coerce')
sdf['Decay Date'] = pd.to_datetime(sdf['Decay Date'], format='%Y %b %d', errors='coerce')

# Create a Year of Launch and Decay
sdf['Launch Year'] = sdf['Launch Date'].dt.year.astype('Int64')
sdf['Decay Year'] = sdf['Decay Date'].dt.year.astype('Int64')
sdf['Launch Month No'] = sdf['Launch Date'].dt.month.astype('Int64')
sdf['Decay Month No'] = sdf['Decay Date'].dt.month.astype('Int64')
sdf['Launch Month'] = sdf['Launch Date'].dt.strftime('%B')
sdf['Decay Month'] = sdf['Decay Date'].dt.strftime('%B')
sdf['Launch Decade'] = ((sdf['Launch Year'] // 10) * 10)
sdf['Decay Decade'] = ((sdf['Decay Year'] // 10) * 10)

# Replace null Decay Date with today's date
#sdf['Effective Decay Date'] = sdf['Decay Date'].fillna(today)
# If Status == "In Orbit", set to today, else use Decay Date
sdf['Effective Decay Date'] = np.where(
    sdf['In Orbit'] == True,
    today,
    sdf['Decay Date']
)

# Convert to datetime 
sdf['Effective Decay Date'] = pd.to_datetime(sdf['Effective Decay Date'], errors='coerce')

# Calculate lifetime
delta = sdf['Effective Decay Date'] - sdf['Launch Date']
sdf['Lifespan Days'] = delta.dt.days
sdf['Lifespan Months'] = (delta.dt.days / 30.44).round(1)  # Approximate month length
sdf['Lifespan Years'] = (delta.dt.days / 365.25).round(2)  # Account for leap years

# Format as dd/mm/yyyy (because we are British)
sdf['Launch Date'] = sdf['Launch Date'].dt.strftime('%d/%m/%Y')
sdf['Decay Date'] = sdf['Decay Date'].dt.strftime('%d/%m/%Y')

# Drop Effective Decay Date column
sdf = sdf.drop('Effective Decay Date', axis=1)

In [89]:
sdf.sample(20)[['CosparID','Type', 'Status', 'Launch Date', 'Decay Date','Lifespan Days','In Orbit']]

,CosparID,Type,Status,Launch Date,Decay Date,Lifespan Days,In Orbit
35007,2009-029A,Payload,Reentered,21/05/2009,23/04/2021,4355.0,False
11511,1979-077B,Rocket Body,Reentered,28/08/1979,28/09/1979,31.0,False
41579,2016-034B,Rocket Body,In orbit,04/06/2016,NaN,3467.0,True
57727,2023-134E,Payload,In orbit,04/09/2023,NaN,819.0,True
10214,1977-065AU,Debris,Reentered,14/07/1977,13/04/1981,1369.0,False
3331,1968-062A,Payload,Landed,30/07/1968,05/08/1968,6.0,False
59116,2024-044T,Payload,In orbit,04/03/2024,NaN,637.0,True
42093,1968-114W,Debris,In orbit,15/12/1968,NaN,20805.0,True
44143,2019-006AF,Debris,Reentered,24/01/2019,08/05/2019,104.0,False
1575,1958 BET 3,Component,In orbit,17/03/1958,NaN,24731.0,True


### Map Country Code to Country Name

The country codes ('State') supplied are two figure codes. 
These are first converted to three figure codes, then a Country name derived from this.

In [90]:
# Convert alpha-2 to alpha-3
def alpha2_to_alpha3(code):
    try:
        return pycountry.countries.get(alpha_2=code).alpha_3
    except:
        return None

# Apply conversion to State column
sdf['Country Code'] = sdf['State'].apply(alpha2_to_alpha3)
sdf['Country'] = sdf['Country Code'].apply(lambda x: pycountry.countries.get(alpha_3=x).name if x else None)

# Operator and Manufacturer Codes
sdf['Operator Country Code 3'] = sdf['Operator Country Code'].apply(alpha2_to_alpha3)
sdf['Manufacturer Country Code 3'] = sdf['Manufacturer Country Code'].apply(alpha2_to_alpha3)



# Drop State Column
sdf.drop(columns=['State', 'Operator Country Code', 'Manufacturer Country Code'], axis=1, inplace=True)
sdf.rename(columns={'Operator Country Code 3': 'Operator Country Code', 'Manufacturer Country Code 3': 'Manufacturer Country Code'}, inplace=True)

In [91]:
# Create a combined "Program Root" field
#sdf["Program Root"] = sdf["Program"].str.split().str[0]
sdf["Program Root"] = sdf["Program"].str.extract(r"^([A-Za-z0-9]+)")

Tidy Apogee and Perigee columns

In [92]:
# Convert to numeric, non-numeric → NaN
sdf['Apogee'] = pd.to_numeric(sdf['Apogee'], errors='coerce')
sdf['Perigee'] = pd.to_numeric(sdf['Perigee'], errors='coerce')

# Flip negatives to positive
sdf['Apogee'] = sdf['Apogee'].abs()
sdf['Perigee'] = sdf['Perigee'].abs()

In [93]:
# Reorder columns alphabetically
sdf = sdf.reindex(sorted(sdf.columns), axis=1)

In [94]:
sdf.columns

Index(['Apogee', 'Class', 'CosparID', 'Country', 'Country Code', 'Decay Date',
       'Decay Decade', 'Decay Month', 'Decay Month No', 'Decay Year',
       'Diameter', 'In Orbit', 'Launch Date', 'Launch Decade', 'Launch Month',
       'Launch Month No', 'Launch Year', 'Length', 'Lifespan Days',
       'Lifespan Months', 'Lifespan Years', 'Manufacturer City',
       'Manufacturer Code', 'Manufacturer Country',
       'Manufacturer Country Code', 'Manufacturer Latitude',
       'Manufacturer Long Name', 'Manufacturer Longitude',
       'Manufacturer Parent', 'Manufacturer Short Name', 'Manufacturer State',
       'Mass', 'Name', 'Operator City', 'Operator Code', 'Operator Country',
       'Operator Country Code', 'Operator Latitude', 'Operator Long Name',
       'Operator Longitude', 'Operator Parent', 'Operator Short Name',
       'Operator State', 'Perigee', 'Primary Category', 'Program',
       'Program Root', 'Secondary Category', 'Shape', 'Span', 'Status',
       'Type'],
      dtyp

In [102]:
sdf[sdf['Operator Country'] == 'United Kingdom'][['Name', 'Launch Date','Class']]

,Name,Launch Date,Class
284,Ariel 1,26/04/1962,Government (Civil)
770,Ariel 2,27/03/1964,Government (Civil)
846,Ariel II despin weight?,27/03/1964,Unknown
2772,Ariel 3,05/05/1967,Government (Civil)
2858,UK-3 despin weight,05/05/1967,Unknown
...,...,...,...
61559,OneWeb SL0707,20/10/2024,Commercial
61560,OneWeb SL0708,20/10/2024,Commercial
63097,Sedna-2,14/01/2025,Commercial
64489,ForgeStar-1,23/06/2025,Commercial


In [96]:
sdf.shape

(66693, 52)

### Write out the cleaned dataset for further use



In [97]:
# Create filenames, one timestamped, one 'latest'
filename = f'{filepath}space_objects_{timestamp_formatted}.csv'
latestname = f'{filepath}space_objects_latest.csv'

# Write out CSV files
sdf.to_csv(filename, index=False)
sdf.to_csv(latestname, index=False)